# RQ3: Predicting flight delays — completed pipeline
This is the original full notebook, with its feature definitions, loading and eight feature comparisons preserved. The tuning section now compares your previous EBM with stronger EBM and CatBoost candidates, chooses a threshold on development data, and completes retrospective evaluation and diagnostics.

**Run top to bottom in the original project environment.** Dependencies include `interpret`, `catboost`, `xgboost`, scikit-learn, pandas, numpy, matplotlib, seaborn and joblib. The cleaned flight parquet and RQ1/RQ2 CSVs are required. No performance results are invented: the winning model is determined when you run the notebook.

Prediction target: arrival ≥15 minutes late, among completed flights, using information intended to be available before departure. Cancelled/diverted flights are outside the target population.

**Evaluation contract:** expanding January–June → July and January–July → August development folds; mean validation AP selects the model, mean validation F1 selects its threshold. September–December are rolling retrospective tests: train January–August → September, January–September → October, January–October → November, and January–November → December. The existing validation winner and threshold are frozen before testing; each test model uses all preceding training rows.

**Snapshot limitation:** historical target encodings use only preceding months, but fixed Jan–Jun RQ1 snapshots can still contain structural look-ahead for early training rows, and RQ2 provenance remains unverified. This notebook cannot reconstruct unavailable upstream snapshots. Results remain a fixed-snapshot experiment until that is resolved.

### Setup and imports

In [ ]:
from pathlib import Path
import sys
import json

from sklearn.model_selection import ParameterGrid

import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
from xgboost import XGBClassifier

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import CLEAN_PARQUET, TABLES_DIR
from src.data_loading import load_flights
from IPython.display import display
import numpy as np
# Preserve source paths; result tables have a separate directory.
SOURCE_TABLES_DIR = Path(TABLES_DIR)
RESULT_TABLES_DIR = PROJECT_DIR / 'results' / 'tables'
RESULT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
FLIGHTS_SRC = "flights_2024_clean.parquet"

BASELINE_COLS = [
    ("route_past_delay_rate", "derived from training data separately for each split"),
]
FLIGHT_COLS = [  # standard features from the original flight dataframe
    ("month", FLIGHTS_SRC),
    ("day_of_week", FLIGHTS_SRC),
    ("sched_hour", f"{FLIGHTS_SRC}: crs_dep_time // 100"),
    ("op_unique_carrier", FLIGHTS_SRC),
    ("distance", FLIGHTS_SRC),
]
RQ1_COLS = [
    ("route_cluster", "rq1_route_clusters_jan_jun.csv (joined on origin, dest, op_unique_carrier)"),
    ("cluster_past_delay_rate", "rq1_cluster_delay_rates_jan_jun.csv: historical_delay_rate (joined on route_cluster)"),
]
RQ2_COLS = [
    ("origin_community", "03_airports.csv: louvain (joined on origin)"),
    ("dest_community", "03_airports.csv: louvain (joined on dest)"),
    ("crosses_community", "derived: origin_community != dest_community"),
    ("origin_betweenness", "03_airports.csv: betweenness (joined on origin)"),
    ("dest_betweenness", "03_airports.csv: betweenness (joined on dest)"),
]
OTHER_COLS = [  # not features: route key for the baseline, and the target
    ("origin", FLIGHTS_SRC),
    ("dest", FLIGHTS_SRC),
    ("late_arrival", f"target, {FLIGHTS_SRC}: arr_delay >= 15"),
]

ROUTE_KEYS = ["origin", "dest", "op_unique_carrier"]

BASELINE_FEATURES = [column for column, _ in BASELINE_COLS]
FLIGHT_FEATURES = [column for column, _ in FLIGHT_COLS]
RQ1_FEATURES = [column for column, _ in RQ1_COLS]
RQ2_FEATURES = [column for column, _ in RQ2_COLS]


In [ ]:
# Delay columns are loaded only to build the target, never as features.
flights = load_flights(
    CLEAN_PARQUET,
    columns=["month", "day_of_week", "op_unique_carrier", "origin", "dest",
             "crs_dep_time", "distance", "status", "arr_delay"],
)

# Cancelled and diverted flights have no arrival delay.
flights = flights[flights["status"] == "completed"].dropna(subset=["arr_delay"]).reset_index(drop=True)

# Late = arrives 15+ minutes late (official FAA definition).
flights["late_arrival"] = (flights["arr_delay"] >= 15).astype(int)
flights["sched_hour"] = ((flights["crs_dep_time"] // 100) % 24).astype(int)
flights = flights.drop(columns=["status", "arr_delay", "crs_dep_time"])

# RQ1 output: route + airline -> cluster, and the cluster's past delay rate.
route_lookup = pd.read_csv(TABLES_DIR / "rq1_route_clusters_jan_jun.csv")
route_rates = pd.read_csv(TABLES_DIR / "rq1_cluster_delay_rates_jan_jun.csv")[
    ["route_cluster", "historical_delay_rate"]
].rename(columns={"historical_delay_rate": "cluster_past_delay_rate"})

flights = flights.merge(route_lookup[ROUTE_KEYS + ["route_cluster"]], on=ROUTE_KEYS, how="left", validate="many_to_one")
flights = flights.merge(route_rates, on="route_cluster", how="left", validate="many_to_one")

# Routes not seen in Jan-Jun have no cluster.
flights["route_cluster"] = flights["route_cluster"].astype("Int64").astype("string").fillna("unknown")

# RQ2 output: community and betweenness of the origin and of the destination airport.
airports = pd.read_csv(TABLES_DIR / "03_airports.csv")[["airport", "louvain", "betweenness"]]
for side in ["origin", "dest"]:
    side_features = airports.rename(columns={
        "airport": side,
        "louvain": f"{side}_community",
        "betweenness": f"{side}_betweenness",
    })
    flights = flights.merge(side_features, on=side, how="left", validate="many_to_one")
    flights[f"{side}_community"] = flights[f"{side}_community"].astype("Int64").astype("string").fillna("unknown")

flights["crosses_community"] = (flights["origin_community"] != flights["dest_community"]).astype(int)

rq3_df = flights[["origin", "dest"] + FLIGHT_FEATURES + RQ1_FEATURES + RQ2_FEATURES + ["late_arrival"]].copy()

print(f"RQ3 modeling dataframe rows: {len(rq3_df):,}")
print(f"Target rate: {rq3_df['late_arrival'].mean():.3%}")
print(f"Unknown route cluster: {(rq3_df['route_cluster'] == 'unknown').mean():.2%}")
print(f"Unknown origin / dest community: {(rq3_df['origin_community'] == 'unknown').mean():.2%} / {(rq3_df['dest_community'] == 'unknown').mean():.2%}")


feature_groups = [
    ("Baseline", BASELINE_COLS),
    ("standard_flight", FLIGHT_COLS),
    ("RQ1", RQ1_COLS),
    ("RQ2", RQ2_COLS),
    ("Other (not features)", OTHER_COLS),
]
for name, cols in feature_groups:
    print(f"{name}")
    for col, src in cols:
        print(f"    {col:<24} <- {src}")

# Dataset columns are accounted for; route keys and the target are not model features.
listed = [c for c, _ in FLIGHT_COLS + RQ1_COLS + RQ2_COLS + OTHER_COLS]
assert sorted(listed) == sorted(rq3_df.columns)


### Feature groups

In [ ]:
for name, features in [
    ("Baseline", BASELINE_FEATURES),
    ("standard_flight", FLIGHT_FEATURES),
    ("RQ1", RQ1_FEATURES),
    ("RQ2", RQ2_FEATURES),
]:
    print(f"{name}: {features}")

### Feature configurations
These eight configurations are preserved exactly. The names `Baseline + RQ1`,
`Baseline + RQ2`, and `Baseline + RQ1 + RQ2` also include `standard_flight`;
inspect the table below for the exact inputs.


In [ ]:
# Helper function to combine features without duplicates
def combine_features(*feature_lists):
    return list(dict.fromkeys(
        col
        for features in feature_lists
        for col in features
    ))


FEATURE_CONFIGS = {
    "Baseline": BASELINE_FEATURES,
    "standard_flight": FLIGHT_FEATURES,
    "RQ1": RQ1_FEATURES,
    "RQ2": RQ2_FEATURES,

    "Baseline + standard_flight": combine_features(
        BASELINE_FEATURES,
        FLIGHT_FEATURES
    ),

    "Baseline + RQ1": combine_features(
        BASELINE_FEATURES,
        FLIGHT_FEATURES,
        RQ1_FEATURES
    ),

    "Baseline + RQ2": combine_features(
        BASELINE_FEATURES,
        FLIGHT_FEATURES,
        RQ2_FEATURES
    ),

    "Baseline + RQ1 + RQ2": combine_features(
        BASELINE_FEATURES,
        FLIGHT_FEATURES,
        RQ1_FEATURES,
        RQ2_FEATURES
    )
}

In [ ]:
assert len(FEATURE_CONFIGS) == 8
feature_config_table = pd.DataFrame([
    {"Feature Config": name, "Number of Features": len(features),
     "Features": ", ".join(features)}
    for name, features in FEATURE_CONFIGS.items()
])
display(feature_config_table)


### Expanding-window feature selection
Each configuration is fitted from scratch twice:

| Fold | Training months | Validation month |
| --- | --- | --- |
| July | 1–6 | 7 |
| August | 1–7 | 8 |


In [ ]:
VALIDATION_FOLDS = (
    {"fold": "July", "train_months": tuple(range(1, 7)), "validation_month": 7},
    {"fold": "August", "train_months": tuple(range(1, 8)), "validation_month": 8},
)
MODEL_TYPE = "EBM"
SELECTION_METRIC = "PR-AUC"
RANDOM_STATE = 42
# None fits every training row. Set e.g. 200_000 for a quicker comparison;
# all configurations then receive the same sample within each fold.
SAMPLE_SIZE = 200_000
THRESHOLD = 0.5


def make_model(model_type=MODEL_TYPE, random_state=RANDOM_STATE):
    if model_type == "EBM":
        return ExplainableBoostingClassifier(random_state=random_state, interactions=0)
    if model_type == "XGBoost":
        return XGBClassifier(
            n_estimators=100, max_depth=6, enable_categorical=True,
            tree_method="hist", random_state=random_state,
            eval_metric="logloss",
        )
    raise ValueError("model_type must be 'EBM' or 'XGBoost'")


def add_historical_rate(
    train, val, keys, output_col, smoothing=20, initial_prior=0.5
):
    """Training: earlier months only. Validation: full training history."""
    train = train.copy()
    val = val.copy()

    if train.empty:
        raise ValueError("Training data must not be empty")
    if train["month"].isna().any() or val["month"].isna().any():
        raise ValueError("Month must not be missing")
    if not val.empty and train["month"].max() >= val["month"].min():
        raise ValueError("Validation must follow the training months")
    if smoothing < 0:
        raise ValueError("smoothing must be non-negative")

    def encode(history, rows):
        if history.empty:
            # Fixed prior: do not estimate it from future outcomes.
            return pd.Series(initial_prior, index=rows.index, dtype=float)

        prior = history["late_arrival"].mean()

        # Unknown cluster membership is not a meaningful shared cluster.
        if keys == ["route_cluster"]:
            history = history.loc[
                history["route_cluster"].notna()
                & history["route_cluster"].ne("unknown")
            ]

        stats = (
            history.groupby(keys, observed=True, dropna=False)["late_arrival"]
            .agg(["sum", "count"])
            .reset_index()
        )
        stats["_rate"] = (
            stats["sum"] + smoothing * prior
        ) / (stats["count"] + smoothing)

        # Left merge preserves row order; restore the original index.
        encoded = rows[keys].merge(
            stats[keys + ["_rate"]],
            on=keys,
            how="left",
            validate="many_to_one",
            sort=False,
        )["_rate"].fillna(prior)

        return pd.Series(encoded.to_numpy(), index=rows.index)

    train[output_col] = float(initial_prior)

    for month in sorted(train["month"].unique()):
        mask = train["month"].eq(month)
        history = train.loc[train["month"] < month]
        train.loc[mask, output_col] = encode(
            history, train.loc[mask]
        )

    val[output_col] = encode(train, val)
    return train, val


def prepare_fold(df, fold, sample_size=None, random_state=42):
    train = df[df["month"].isin(fold["train_months"])].copy()
    val = df[df["month"] == fold["validation_month"]].copy()
    if train.empty or val.empty:
        raise ValueError(f"Empty training or validation data for {fold['fold']}")
    missing_months = set(fold["train_months"]) - set(train["month"].unique())
    if missing_months:
        raise ValueError(f"Missing training months: {sorted(missing_months)}")
    if train["late_arrival"].nunique() != 2 or val["late_arrival"].nunique() != 2:
        raise ValueError("Both target classes are required in training and validation")
    train, val = add_historical_rate(train, val, ROUTE_KEYS, "route_past_delay_rate")
    train, val = add_historical_rate(train, val, ["route_cluster"], "cluster_past_delay_rate")
    n_available = len(train)
    if sample_size is not None:
        if sample_size < 2:
            raise ValueError("sample_size must be at least 2 or None")
        if len(train) > sample_size:
            train = train.sample(n=sample_size, random_state=random_state)
    if train["late_arrival"].nunique() != 2:
        raise ValueError("Training sample must contain both target classes")
    # Learn categorical vocabularies from training only. Unseen validation
    # categories become missing values, supported by both estimators.
    all_features = list(dict.fromkeys(f for fs in FEATURE_CONFIGS.values() for f in fs))
    for col in all_features:
        if (pd.api.types.is_object_dtype(train[col])
                or isinstance(train[col].dtype, (pd.StringDtype, pd.CategoricalDtype))):
            categories = pd.Index(train[col].dropna().unique())
            dtype = pd.CategoricalDtype(categories=categories)
            train[col] = train[col].astype(dtype)
            val[col] = val[col].astype(dtype)
    return train, val, n_available


def evaluate_model(model, X, y, threshold=0.5):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y, probs),
        "PR-AUC": average_precision_score(y, probs),
        "Log Loss": log_loss(y, probs, labels=[0, 1]),
        "Brier Score": brier_score_loss(y, probs),
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds, zero_division=0),
        "Recall": recall_score(y, preds, zero_division=0),
        "F1": f1_score(y, preds, zero_division=0),
    }


def run_feature_selection(df, model_type=MODEL_TYPE, sample_size=SAMPLE_SIZE):
    rows = []
    for fold in VALIDATION_FOLDS:
        train, val, n_available = prepare_fold(df, fold, sample_size, RANDOM_STATE)
        for name, features in FEATURE_CONFIGS.items():
            print(f"{model_type} | {name} | train 1–{max(fold['train_months'])} → validate {fold['validation_month']}")
            model = make_model(model_type, RANDOM_STATE)
            start = time.perf_counter()
            model.fit(train[features], train["late_arrival"])
            duration = time.perf_counter() - start
            for split, data in (("Train", train), ("Validation", val)):
                rows.append({
                    "Estimator": model_type, "Feature Config": name,
                    "Number of Features": len(features), "Fold": fold["fold"],
                    "Train Period": f"1–{max(fold['train_months'])}",
                    "Validation Month": fold["validation_month"], "Split": split,
                    "Available Train Samples": n_available, "Train Samples": len(train),
                    "Eval Samples": len(data), "Training Seconds": duration,
                    **evaluate_model(model, data[features], data["late_arrival"], THRESHOLD),
                })
            # Only results are retained, avoiding 16 large fitted models in memory.
    results = pd.DataFrame(rows)
    assert len(results) == len(FEATURE_CONFIGS) * len(VALIDATION_FOLDS) * 2
    return results


In [ ]:
# Fit the eight original feature configurations on identical chronological folds.
results_df = run_feature_selection(rq3_df, model_type=MODEL_TYPE, sample_size=SAMPLE_SIZE)
validation_results = results_df[results_df["Split"] == "Validation"].copy()
display(validation_results.round(4))


### Rank the feature configurations
Rank feature configurations by mean validation PR-AUC across the two groups

All other metrics, fold variability, and the train–validation gap are reported for comparison but do not affect the ranking.


In [ ]:
SELECTION_METRIC = "PR-AUC"

METRICS = [
    "PR-AUC", "ROC-AUC", "Log Loss", "Brier Score",
    "Accuracy", "Precision", "Recall", "F1"
]

means = (
    validation_results.groupby("Feature Config", sort=False)[METRICS]
    .mean()
    .add_prefix("Mean Validation ")
)

per_fold = validation_results.pivot(
    index="Feature Config",
    columns="Validation Month",
    values=SELECTION_METRIC,
).rename(columns={7: "July PR-AUC", 8: "August PR-AUC"})

ranking = means.join(per_fold)

ranking["Validation PR-AUC SD"] = (
    validation_results.groupby("Feature Config")[SELECTION_METRIC].std()
)
ranking["Number of Features"] = pd.Series({
    name: len(features) for name, features in FEATURE_CONFIGS.items()
})

train_means = (
    results_df[results_df["Split"] == "Train"]
    .groupby("Feature Config")[SELECTION_METRIC]
    .mean()
)
ranking["Mean Train PR-AUC"] = train_means
ranking["Train–Validation PR-AUC Gap"] = (
    train_means - ranking["Mean Validation PR-AUC"]
)

ranking = (
    ranking.reset_index()
    .sort_values(
        ["Mean Validation PR-AUC", "Number of Features", "Feature Config"],
        ascending=[False, True, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
ranking.insert(0, "Rank", range(1, len(ranking) + 1))

best_config = ranking.loc[0, "Feature Config"]
best_features = FEATURE_CONFIGS[best_config].copy()

print(f"Selected features for {MODEL_TYPE}: {best_config}")
print(f"Mean validation PR-AUC: {ranking.loc[0, 'Mean Validation PR-AUC']:.4f}")
print(f"Features: {best_features}")

display(ranking.round(4))

In [ ]:

selection_prefix = f"rq3_{MODEL_TYPE.lower()}_feature_selection"
results_df.to_csv(RESULT_TABLES_DIR / f"{selection_prefix}_fold_scores.csv", index=False)
ranking.to_csv(RESULT_TABLES_DIR / f"{selection_prefix}_ranking.csv", index=False)
selection = {
    "estimator": MODEL_TYPE, "feature_config": best_config, "features": best_features,
    "selection_metric": SELECTION_METRIC,
    "aggregation": "unweighted mean over validation months 7 and 8",
    "mean_validation_score": float(ranking.loc[0, "Mean Validation PR-AUC"]),
    "sample_size": SAMPLE_SIZE, "random_state": RANDOM_STATE,
    "folds": list(VALIDATION_FOLDS), "reserved_test_months": [9, 10, 11, 12],
}
(RESULT_TABLES_DIR / f"{selection_prefix}_selected.json").write_text(json.dumps(selection, indent=2))
print(f"Saved {selection_prefix} results and selected features to {RESULT_TABLES_DIR}")


In [ ]:
# Same configuration order in both panels; highlight the validation-selected winner.
order = ranking["Feature Config"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharey=True)
for ax, split in zip(axes, ["Train", "Validation"]):
    for month, marker in [(7, "o"), (8, "s")]:
        scores = results_df[(results_df["Split"] == split) & (results_df["Validation Month"] == month)].set_index("Feature Config")
        ax.plot(range(len(order)), scores.loc[order, SELECTION_METRIC], marker=marker,
                label=f"Train 1–{month - 1}, validate {month}")
    best_index = order.index(best_config)
    ax.axvspan(best_index - 0.3, best_index + 0.3, color="gold", alpha=0.25,
               label="Selected by mean validation PR-AUC")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=40, ha="right")
    ax.set_title(f"{MODEL_TYPE}: {split} PR-AUC")
    ax.set_ylabel("Average precision (higher is better)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
metrics = [
    "Mean Validation PR-AUC",
    "Mean Validation ROC-AUC",
    "Mean Validation Log Loss",
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric in zip(axes, metrics):
    sns.barplot(
        data=ranking,
        x="Feature Config",
        y=metric,
        order=ranking["Feature Config"],
        ax=ax,
    )

    values = ranking[metric]
    low, high = values.min(), values.max()
    pad = max((high - low) * 0.2, abs(low) * 0.02, 0.001)
    ax.set_ylim(low - pad, high + pad)

    # Mark the truncated y-axis.
    d = 0.012
    ax.plot((-d, d), (-d, d), transform=ax.transAxes,
            color="black", clip_on=False, linewidth=1)
    ax.plot((-d, d), (d, 3 * d), transform=ax.transAxes,
            color="black", clip_on=False, linewidth=1)

    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)

fig.tight_layout()
plt.show()

## Stronger model search
The original feature comparison is descriptive. The main search retains every original feature in its `all` candidates and explicitly tests calendar alternatives. It includes your previous tuned EBM, so an improvement is measured against the same folds and sample size. CatBoost handles categorical variables directly and can learn higher-order interactions; whether it helps must be established by validation.

Model selection uses AP rather than recall at an arbitrary threshold. Threshold selection happens separately. The preceding version accidentally ranked features by recall despite documenting AP; that inconsistency is corrected above.


In [ ]:
from pathlib import Path
import sys, json, time, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.metrics import (average_precision_score, roc_auc_score, log_loss,
    brier_score_loss, precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, precision_recall_curve)
from sklearn.calibration import calibration_curve
from interpret.glassbox import ExplainableBoostingClassifier
from catboost import CatBoostClassifier
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks': PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))
from src.config import CLEAN_PARQUET, TABLES_DIR
from src.data_loading import load_flights
SOURCE_TABLES = SOURCE_TABLES_DIR
OUT = PROJECT_DIR / 'results'
for name in ['tables', 'figures', 'logs', 'models']:
    (OUT / name).mkdir(parents=True, exist_ok=True)
SEED = 42
SAMPLE_SIZE = 200_000
DEV_MONTHS = (7, 8)
TEST_MONTHS = (9, 10, 11, 12)
SNAPSHOT_LAST_MONTH = 0
# Fill these only after checking the upstream construction, not just the filenames.
SNAPSHOT_AUDIT_NOTE = 'Fixed Jan–Jun RQ1 snapshot; RQ2 provenance unverified. Earlier training rows may contain structural look-ahead.'
THRESHOLDS = np.arange(0.05, 0.601, 0.01)
# Selection criterion fixed before running: equal-month mean AP, threshold by mean F1.
def save_table(df, name):
    df.to_csv(OUT / 'tables' / f'rq3_{name}.csv', index=False)
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / f'rq3_{name}.png', dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
def add_historical_rate(
    train, val, keys, output_col, smoothing=20, initial_prior=0.5
):
    """Training: earlier months only. Validation: full training history."""
    train = train.copy()
    val = val.copy()

    if train.empty:
        raise ValueError("Training data must not be empty")
    if train["month"].isna().any() or val["month"].isna().any():
        raise ValueError("Month must not be missing")
    if not val.empty and train["month"].max() >= val["month"].min():
        raise ValueError("Validation must follow the training months")
    if smoothing < 0:
        raise ValueError("smoothing must be non-negative")

    def encode(history, rows):
        if history.empty:
            # Fixed prior: do not estimate it from future outcomes.
            return pd.Series(initial_prior, index=rows.index, dtype=float)

        prior = history["late_arrival"].mean()

        # Unknown cluster membership is not a meaningful shared cluster.
        if keys == ["route_cluster"]:
            history = history.loc[
                history["route_cluster"].notna()
                & history["route_cluster"].ne("unknown")
            ]

        stats = (
            history.groupby(keys, observed=True, dropna=False)["late_arrival"]
            .agg(["sum", "count"])
            .reset_index()
        )
        stats["_rate"] = (
            stats["sum"] + smoothing * prior
        ) / (stats["count"] + smoothing)

        # Left merge preserves row order; restore the original index.
        encoded = rows[keys].merge(
            stats[keys + ["_rate"]],
            on=keys,
            how="left",
            validate="many_to_one",
            sort=False,
        )["_rate"].fillna(prior)

        return pd.Series(encoded.to_numpy(), index=rows.index)

    train[output_col] = float(initial_prior)

    for month in sorted(train["month"].unique()):
        mask = train["month"].eq(month)
        history = train.loc[train["month"] < month]
        train.loc[mask, output_col] = encode(
            history, train.loc[mask]
        )

    val[output_col] = encode(train, val)
    return train, val

In [ ]:
def make_history(df):
    result = df.copy()
    empty = result.iloc[:0].copy()
    for keys, col in [(ROUTE_KEYS, 'route_past_delay_rate'),
                      (['route_cluster'], 'cluster_past_delay_rate')]:
        result, _ = add_historical_rate(result, empty, keys, col)
    return result

# Test future-label invariance and same-month exclusion using a tiny known example.
toy = pd.DataFrame({'month': [1, 1, 2, 2, 3], 'origin': ['A']*5,
    'dest': ['B']*5, 'op_unique_carrier': ['C']*5,
    'route_cluster': ['0']*5, 'late_arrival': [0, 1, 0, 1, 1]})
a = make_history(toy)
changed = toy.copy(); changed.loc[changed.month >= 2, 'late_arrival'] = 1
b = make_history(changed)
for col in ['route_past_delay_rate', 'cluster_past_delay_rate']:
    np.testing.assert_allclose(a.loc[a.month <= 2, col], b.loc[b.month <= 2, col])
    assert a.loc[a.month == 2, col].nunique() == 1
    assert a.loc[a.month == 2, col].iloc[0] == 0.5
print('Historical feature invariance checks passed.')
data = make_history(rq3_df)
for col, period in [('month', 12), ('sched_hour', 24), ('day_of_week', 7)]:
    data[col + '_sin'] = np.sin(2*np.pi*data[col]/period)
    data[col + '_cos'] = np.cos(2*np.pi*data[col]/period)

STANDARD = FLIGHT_FEATURES
HISTORY = ['route_past_delay_rate']
GROUPS = {
    'standard': STANDARD,
    'base': HISTORY + STANDARD,
    'base_rq1': HISTORY + STANDARD + RQ1_FEATURES,
    'base_rq2': HISTORY + STANDARD + RQ2_FEATURES,
    'all': HISTORY + STANDARD + RQ1_FEATURES + RQ2_FEATURES,
}
CYCLIC = [f'{c}_{trig}' for c in ['month', 'sched_hour', 'day_of_week'] for trig in ['sin', 'cos']]
GROUPS['all_cyclic'] = [c for c in GROUPS['all'] if c != 'month'] + CYCLIC
GROUPS['all_no_month'] = [c for c in GROUPS['all'] if c != 'month']
CATEGORICAL = ['op_unique_carrier', 'route_cluster', 'origin_community', 'dest_community']
assert not set(GROUPS['all']) & {'late_arrival', 'arr_delay', 'dep_delay'}

def split_month(month, size=SAMPLE_SIZE):
    train = data.loc[data.month.between(SNAPSHOT_LAST_MONTH+1, month-1)].copy()
    test = data.loc[data.month.eq(month)].copy()
    assert len(train) and len(test) and train.month.max() < test.month.min()
    if size and len(train) > size: train = train.sample(size, random_state=SEED)
    assert train.late_arrival.nunique() == test.late_arrival.nunique() == 2
    return train, test

def inputs(train, test, features):
    x, z = train[features].copy(), test[features].copy()
    audit = []
    for col in features:
        if col in CATEGORICAL:
            x[col] = x[col].astype('string').fillna('__missing__').astype(str)
            z[col] = z[col].astype('string').fillna('__missing__').astype(str)
            unseen = ~z[col].isin(x[col].unique())
            audit.append({'feature': col, 'unseen_fraction': unseen.mean(),
                          'missing_fraction': test[col].isna().mean()})
            z.loc[unseen, col] = '__unknown__'
        else:
            fill = pd.to_numeric(x[col], errors='coerce').median()
            fill = 0.0 if pd.isna(fill) else fill
            x[col] = pd.to_numeric(x[col], errors='coerce').fillna(fill).astype(float)
            z[col] = pd.to_numeric(z[col], errors='coerce').fillna(fill).astype(float)
            audit.append({'feature': col, 'unseen_fraction': np.nan,
                          'missing_fraction': test[col].isna().mean()})
    return x, z, audit

## 4. Models and fair comparisons
Compare constant and route-rate baselines, an additive EBM, an interaction EBM and CatBoost. Every candidate receives the same reproducible training rows in a fold; evaluation uses the full month. CatBoost uses a fixed iteration budget: development labels are not used for early stopping. No resampling or class weighting changes the probability prior.

Feature ablations answer the RQ1/RQ2 question within a model family. Calendar variants test the month hypothesis. A modest search is intentional; broader tuning should follow diagnostics, not the retrospective test results.

In [ ]:
# Keep all original features for the main search; calendar variants are controlled ablations.
CANDIDATES = [
    {'id':'constant','kind':'constant','group':'base','params':{}},
    {'id':'route_rate','kind':'route','group':'base','params':{}},
    {'id':'previous_ebm','kind':'ebm','group':'all',
     'params':{'interactions':3,'learning_rate':0.015,'max_leaves':2,'min_samples_leaf':4}},
]
for group in ['all','all_cyclic','all_no_month']:
    for depth, leaves in [(4,20),(6,50),(8,100)]:
        CANDIDATES.append({'id':f'catboost_d{depth}__{group}','kind':'catboost','group':group,
            'params':{'depth':depth,'iterations':700,'learning_rate':0.04,'l2_leaf_reg':leaves}})
    for interactions in [0,10]:
        CANDIDATES.append({'id':f'ebm_i{interactions}__{group}','kind':'ebm','group':group,
            'params':{'interactions':interactions,'learning_rate':0.02,'max_leaves':3,'min_samples_leaf':20}})
# Matched feature ablations at a fixed CatBoost setting.
for group in ['standard','base','base_rq1','base_rq2']:
    CANDIDATES.append({'id':f'catboost_d6__{group}','kind':'catboost','group':group,
        'params':{'depth':6,'iterations':700,'learning_rate':0.04,'l2_leaf_reg':50}})

def fit_predict(spec, train, test):
    if spec['kind'] == 'constant':
        return np.full(len(test), train.late_arrival.mean()), None, []
    if spec['kind'] == 'route':
        return test.route_past_delay_rate.to_numpy(), None, []
    features = GROUPS[spec['group']]
    x, z, audit = inputs(train, test, features)
    if spec['kind'].startswith('ebm'):
        model = ExplainableBoostingClassifier(random_state=SEED, n_jobs=-1,
            feature_types=['nominal' if f in CATEGORICAL else 'continuous' for f in features], **spec['params'])
    else:
        model = CatBoostClassifier(random_seed=SEED, verbose=False, allow_writing_files=False,
            loss_function='Logloss', cat_features=[c for c in features if c in CATEGORICAL],
            **spec['params'])
    model.fit(x, train.late_arrival)
    return model.predict_proba(z)[:, 1], model, audit

def metrics(y, p, threshold):
    p = np.clip(np.asarray(p), 1e-7, 1-1e-7)
    pred = p >= threshold
    prevalence = np.mean(y)
    return {'AP': average_precision_score(y,p), 'ROC_AUC': roc_auc_score(y,p),
        'Log_loss': log_loss(y,p,labels=[0,1]), 'Brier': brier_score_loss(y,p),
        'Precision': precision_score(y,pred,zero_division=0),
        'Recall': recall_score(y,pred,zero_division=0), 'F1': f1_score(y,pred,zero_division=0),
        'Accuracy': accuracy_score(y,pred), 'Prevalence': prevalence,
        'AP_lift': average_precision_score(y,p)/prevalence,
        'Predicted_positive_rate': pred.mean(), 'Mean_probability': p.mean()}

rows, dev_predictions = [], {}
for month in DEV_MONTHS:
    train, val = split_month(month)
    for spec in CANDIDATES:
        start = time.perf_counter()
        p, _, _ = fit_predict(spec, train, val)
        dev_predictions[(spec['id'], month)] = (val.late_arrival.to_numpy(), p)
        rows.append({'Candidate': spec['id'], 'Month': month,
            'Seconds': time.perf_counter()-start, **metrics(val.late_arrival,p,0.3)})
        print(month, spec['id'], round(rows[-1]['AP'],4), flush=True)
    save_table(pd.DataFrame(rows), 'development_metrics')
dev = pd.DataFrame(rows)
ranking = dev.groupby('Candidate')[['AP','ROC_AUC','Log_loss','Seconds']].mean().sort_values(
    ['AP','Log_loss'], ascending=[False,True])
display(ranking)
save_table(ranking.reset_index(), 'development_ranking')
best_id = ranking.index[0]
best = next(s for s in CANDIDATES if s['id'] == best_id)
# Retain the strongest model using both RQ1 and RQ2 even if a baseline wins overall.
full_ids = [s['id'] for s in CANDIDATES if s['group'] in ['all','all_cyclic','all_no_month']]
full_id = ranking.loc[ranking.index.isin(full_ids)].index[0]
selected_ids = list(dict.fromkeys([best_id, full_id, 'previous_ebm', 'constant', 'route_rate']))
print('Overall winner:', best_id, '| strongest combined-feature model:', full_id)
comparison = ranking.loc[[best_id, 'previous_ebm']].copy()
print('Validation AP improvement over your previous EBM:',
      ranking.loc[best_id,'AP'] - ranking.loc['previous_ebm','AP'])
display(comparison)


## 5. Threshold selection and feature value
Thresholds maximise equal-month mean F1 on development predictions only; ties prefer the higher threshold. No threshold is adjusted using September–December. AP denotes average precision (the PR summary used throughout). AP lift provides context against prevalence, but does not eliminate all effects of prevalence shifts.

In [ ]:
threshold_rows, selected_thresholds = [], {}
for cid in selected_ids:
    for t in THRESHOLDS:
        scores = [metrics(*dev_predictions[(cid,m)], t) for m in DEV_MONTHS]
        threshold_rows.append({'Candidate': cid, 'Threshold': t,
            **{k: np.mean([r[k] for r in scores]) for k in ['F1','Recall','Precision']}})
    curve = pd.DataFrame([r for r in threshold_rows if r['Candidate']==cid])
    selected_thresholds[cid] = float(curve.sort_values(['F1','Threshold'], ascending=False).iloc[0].Threshold)
threshold_table = pd.DataFrame(threshold_rows)
save_table(threshold_table, 'threshold_selection')
display(pd.Series(selected_thresholds, name='Frozen threshold'))
ablations = []
for kind in ['catboost_d6']:
    for month in DEV_MONTHS:
        subset = dev.set_index(['Candidate','Month'])
        base_ap = subset.loc[(kind+'__base',month),'AP']
        for group in ['base_rq1','base_rq2','all','all_cyclic','all_no_month']:
            ap = subset.loc[(kind+'__'+group,month),'AP']
            ablations.append({'Model':kind,'Month':month,'Features':group,'AP':ap,'Delta_AP_vs_base':ap-base_ap})
ablation = pd.DataFrame(ablations)
display(ablation)
save_table(ablation, 'feature_ablation')
fig, ax = plt.subplots(figsize=(10,4))
sns.barplot(data=ablation, x='Features', y='Delta_AP_vs_base', hue='Model', errorbar=None, ax=ax)
ax.axhline(0,color='black',linewidth=1)
save_fig(fig,'feature_ablation')
# Preserve the original 0.3 operating point for an apples-to-apples reference.
original_operating_point = pd.DataFrame([
    {'Candidate':'previous_ebm','Month':m,'Threshold':0.3,
     **metrics(*dev_predictions[('previous_ebm',m)],0.3)} for m in DEV_MONTHS])
save_table(original_operating_point,'previous_ebm_original_threshold_validation')
display(original_operating_point)


## 6. Sample-size sensitivity (development only)
This diagnostic does not change the frozen selection automatically. The 200,000 sample is reused from the search; other sizes refit the strongest combined-feature candidate. If changing the selected protocol afterward, document it before retrospective evaluation.

In [ ]:
sensitivity = []
full_spec = next(s for s in CANDIDATES if s['id']==full_id)
for size in [50_000, 200_000, 500_000]:
    for month in DEV_MONTHS:
        tr, va = split_month(month,size)
        if size == SAMPLE_SIZE:
            y,p = dev_predictions[(full_id,month)]
        else:
            p,_,_ = fit_predict(full_spec,tr,va); y=va.late_arrival
        sensitivity.append({'Requested_size':size,'Actual_size':len(tr),'Month':month,
                            **metrics(y,p,selected_thresholds[full_id])})
save_table(pd.DataFrame(sensitivity),'sample_sensitivity')
display(pd.DataFrame(sensitivity))

## 7. Test the selected model on months 9–12
Keep the model specification, features and threshold selected on July/August validation. Fit a fresh model on **all rows** from months 1–8, 1–9, 1–10 and 1–11, respectively; predict every flight in month 9, 10, 11 and 12. No new model search or threshold tuning takes place in this section.

The upload contains no saved selection output, so `TEST_MODEL_ID = best_id` uses the existing selection when run. If your chosen candidate differs, set its exact candidate ID before running this section. Earlier test-month outcomes may enter subsequent training windows. Historical rates use only preceding months; the previously documented fixed structural-snapshot limitation still applies.

Predictions, metrics and percentage confusion tables are saved under `results/tables`; plots under `results/figures`; settings under `results/logs`.

In [ ]:
import copy
from sklearn.metrics import roc_curve

TEST_MONTHS = (9, 10, 11, 12)
TEST_MODEL_ID = best_id
TEST_SPEC = copy.deepcopy(next(s for s in CANDIDATES if s['id'] == TEST_MODEL_ID))
TEST_THRESHOLD = float(selected_thresholds[TEST_MODEL_ID])
TEST_FEATURES = list(GROUPS[TEST_SPEC['group']])
TEST_SAMPLE_SIZE = None  # All rows from January through the preceding month.
print('Frozen model:', TEST_MODEL_ID, '| threshold:', TEST_THRESHOLD)
display(pd.DataFrame({'Training months': ['1–8', '1–9', '1–10', '1–11'],
                      'Test month': TEST_MONTHS}))
protocol = {'model': TEST_SPEC, 'features': TEST_FEATURES, 'threshold': TEST_THRESHOLD,
    'sample_size': TEST_SAMPLE_SIZE, 'seed': SEED,
    'folds': [{'train_months': list(range(1, m)), 'test_month': m} for m in TEST_MONTHS],
    'selection_months': list(DEV_MONTHS), 'snapshot_audit': SNAPSHOT_AUDIT_NOTE,
    'limitation': 'Retrospective evaluation with fixed structural snapshots.'}
(OUT/'logs'/'rq3_rolling_test_settings.json').write_text(json.dumps(protocol, indent=2))

results, prediction_frames, audit_rows, importance_rows = [], [], [], []
for month in TEST_MONTHS:
    train = data.loc[data.month.between(1, month-1)].copy()
    test = data.loc[data.month.eq(month)].copy()
    assert set(train.month.unique()) == set(range(1, month)), 'Missing training months'
    assert len(test) and train.month.max() < test.month.min()
    assert train.late_arrival.nunique() == test.late_arrival.nunique() == 2
    print(f'Train 1–{month-1}: {len(train):,} rows → test {month}: {len(test):,} rows', flush=True)
    start = time.perf_counter()
    p, model, audit = fit_predict(TEST_SPEC, train, test)
    predicted = (p >= TEST_THRESHOLD).astype(int)
    results.append({'Candidate': TEST_MODEL_ID, 'Month': month,
        'Train_period': f'1–{month-1}', 'Train_N': len(train), 'N': len(test),
        'Threshold': TEST_THRESHOLD, 'Seconds': time.perf_counter()-start,
        **metrics(test.late_arrival, p, TEST_THRESHOLD)})
    frame = test[['origin', 'dest', 'op_unique_carrier']].copy().reset_index(drop=True)
    frame.insert(0, 'Row_id', test.index.to_numpy())
    frame['Month'] = month
    frame['Actual'] = test.late_arrival.to_numpy()
    frame['Probability'] = p
    frame['Predicted'] = predicted
    frame['Threshold'] = TEST_THRESHOLD
    frame['Candidate'] = TEST_MODEL_ID
    prediction_frames.append(frame)
    frame.to_csv(OUT/'tables'/f'rq3_predictions_month_{month}.csv.gz', index=False, compression='gzip')
    audit_rows.extend([{'Month': month, **r} for r in audit])
    if model is not None:
        names = model.term_names_ if TEST_SPEC['kind'].startswith('ebm') else TEST_FEATURES
        values = model.term_importances() if TEST_SPEC['kind'].startswith('ebm') else model.feature_importances_
        importance_rows.extend([{'Month': month, 'Feature': name, 'Importance': float(value)}
                                for name, value in zip(names, values)])
    save_table(pd.DataFrame(results), 'rolling_test_metrics')
    # Release each fitted model after predictions to limit peak memory.
    del model, train, test

result = pd.DataFrame(results)
predictions = pd.concat(prediction_frames, ignore_index=True)
importance = pd.DataFrame(importance_rows, columns=['Month', 'Feature', 'Importance'])
save_table(pd.DataFrame(audit_rows), 'rolling_test_input_audit')
save_table(importance, 'rolling_test_importance')
metric_cols = list(metrics(np.array([0, 1]), np.array([0.2, 0.8]), 0.5))
monthly = result[['Month', 'N'] + metric_cols].copy()
monthly['Month'] = monthly['Month'].astype(str)
mean_row = {'Month': 'Mean (equal month)', 'N': np.nan,
            **result[metric_cols].mean().to_dict()}
pooled_row = {'Month': 'Pooled (all flights)', 'N': len(predictions),
             **metrics(predictions.Actual, predictions.Probability, TEST_THRESHOLD)}
metrics_table = pd.concat([monthly, pd.DataFrame([mean_row, pooled_row])], ignore_index=True)
save_table(metrics_table, 'rolling_test_metrics_summary')
display(metrics_table.round(4))

## 8. Test plots and percentage confusion matrices
All plots describe the frozen selected model. **Row percentages** sum to 100% across each actual class; the delayed diagonal is recall. **Column percentages** sum to 100% within each predicted class; the predicted-delayed diagonal is precision. Empty rows/columns are marked N/A. Each cell also shows its flight count. Here “on time” means arrival delay below the 15-minute target threshold.

AP is average precision, used as the PR-AUC summary. The table includes each month, an equal-month mean and pooled metrics recalculated from all predictions. Feature importance is model-specific and does not establish causality.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, metric in zip(axes.flat, ['AP', 'ROC_AUC', 'Log_loss', 'Precision', 'Recall', 'F1']):
    ax.plot(result.Month, result[metric], marker='o')
    ax.set(title='PR-AUC (average precision)' if metric == 'AP' else metric.replace('_', ' '),
           xlabel='Test month', xticks=TEST_MONTHS)
    ax.grid(alpha=.25)
    if metric != 'Log_loss': ax.set_ylim(0, 1)
fig.suptitle(f'{TEST_MODEL_ID} | fixed threshold {TEST_THRESHOLD:.2f}')
save_fig(fig, 'rolling_test_metrics')

confusions = []
for normalization, axis, title in [('row', 1, 'Within actual class (recall)'),
                                    ('column', 0, 'Within predicted class (precision)')]:
    fig, axes = plt.subplots(1, len(TEST_MONTHS), figsize=(18, 4.5))
    for ax, month in zip(axes, TEST_MONTHS):
        d = predictions.loc[predictions.Month.eq(month)]
        cm = confusion_matrix(d.Actual, d.Predicted, labels=[0, 1])
        denominator = cm.sum(axis=axis, keepdims=True)
        percent = np.divide(cm * 100., denominator, out=np.full((2, 2), np.nan), where=denominator != 0)
        sns.heatmap(np.nan_to_num(percent), vmin=0, vmax=100, cmap='Blues', cbar=False,
                    xticklabels=['On time', 'Delayed'], yticklabels=['On time', 'Delayed'], ax=ax)
        for actual in range(2):
            for predicted in range(2):
                value = percent[actual, predicted]
                label = 'N/A' if np.isnan(value) else f'{value:.1f}%'
                ax.text(predicted+.5, actual+.5, f'{label}\n(n={cm[actual, predicted]:,})',
                        ha='center', va='center', color='white' if value > 50 else 'black')
                confusions.append({'Month': month, 'Normalization': normalization,
                    'Actual': actual, 'Predicted': predicted,
                    'Count': int(cm[actual, predicted]), 'Percentage': value})
        ax.set(title=f'Month {month}', xlabel='Predicted', ylabel='Actual')
    fig.suptitle(title)
    save_fig(fig, f'rolling_test_confusion_{normalization}_percent')
save_table(pd.DataFrame(confusions), 'rolling_test_confusion_percentages')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for month in TEST_MONTHS:
    d = predictions.loc[predictions.Month.eq(month)]
    precision, recall, _ = precision_recall_curve(d.Actual, d.Probability)
    fpr, tpr, _ = roc_curve(d.Actual, d.Probability)
    row = result.set_index('Month').loc[month]
    line, = axes[0].plot(recall, precision, label=f"{month}: AP={row.AP:.3f}")
    axes[0].axhline(d.Actual.mean(), color=line.get_color(), linestyle=':', alpha=.4)
    axes[0].scatter(row.Recall, row.Precision, color=line.get_color(), s=35)
    axes[1].plot(fpr, tpr, label=f"{month}: AUC={row.ROC_AUC:.3f}")
axes[0].set(title='Precision–recall (dots: frozen threshold; dotted: prevalence)', xlabel='Recall', ylabel='Precision')
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set(title='ROC curves', xlabel='False positive rate', ylabel='True positive rate')
for ax in axes:
    ax.legend(title='Test month'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
save_fig(fig, 'rolling_test_pr_roc_curves')

fig, axes = plt.subplots(2, len(TEST_MONTHS), figsize=(18, 8))
for j, month in enumerate(TEST_MONTHS):
    d = predictions.loc[predictions.Month.eq(month)]
    observed, predicted = calibration_curve(d.Actual, d.Probability, n_bins=10, strategy='quantile')
    axes[0, j].plot(predicted, observed, marker='o')
    axes[0, j].plot([0, 1], [0, 1], '--', color='grey')
    axes[0, j].set(title=f'Month {month}', xlabel='Mean predicted probability',
                   ylabel='Observed delayed fraction', xlim=(0, 1), ylim=(0, 1))
    for label, name in [(0, 'On time'), (1, 'Delayed')]:
        axes[1, j].hist(d.loc[d.Actual.eq(label), 'Probability'], bins=np.linspace(0, 1, 41),
                        density=True, alpha=.5, label=name)
    axes[1, j].axvline(TEST_THRESHOLD, color='black', linestyle='--', label='Threshold')
    axes[1, j].set(xlabel='Predicted delay probability', ylabel='Density')
    axes[1, j].legend()
save_fig(fig, 'rolling_test_calibration_probabilities')

if not importance.empty:
    top = importance.groupby('Feature').Importance.mean().nlargest(15).index
    matrix = importance.pivot(index='Feature', columns='Month', values='Importance').reindex(top)
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    matrix.mean(axis=1).sort_values().plot.barh(ax=axes[0])
    axes[0].set(title='Top 15 terms/features: mean importance', xlabel='Model-specific importance')
    sns.heatmap(matrix, cmap='YlGnBu', ax=axes[1])
    axes[1].set(title='Importance across test months', xlabel='Test month', ylabel='')
    save_fig(fig, 'rolling_test_feature_importance')
else:
    print('Selected baseline does not expose model feature importance.')

## 9. Reading the results
Compare September–December AP, ROC-AUC and log loss to assess ranking and probability quality. Use precision, recall and F1 to assess the frozen threshold. Row-normalised confusion matrices show missed delays; column-normalised matrices show how often delay alerts are correct. Calibration plots show whether probabilities match observed frequencies.

This section refits the chosen specification on expanding training windows, without changing its hyperparameters, features or threshold based on test results. The input notebook contains no saved fitted winner, and the project data is not embedded: run in the original project environment to produce actual predictions and plots. Fixed RQ1/RQ2 snapshot limitations remain as documented above.